In [35]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

import imageio.v3 as iio
import os
from PIL import Image
import numpy as np
from tqdm import tqdm
import cv2


In [ ]:
# --- parameters ---
DATA_PATH = "/Volumes/LabData/BMED6460/beetle-master/data/"
IMG_PATCH_DIR = DATA_PATH + "images/development/output/beetle-patches/img_patches-2048"
MASK_PATCH_DIR = DATA_PATH + "images/development/output/beetle-patches/mask_patches-2048"

# out_dir = DATA_PATH + "images/development/output"
OUT_IMG_DIR = DATA_PATH + "images/development/output/beetle-patches/img_patches-256"
OUT_MASK_DIR = DATA_PATH + "images/development/output/beetle-patches/mask_patches-256"
os.makedirs(OUT_IMG_DIR, exist_ok=True)
os.makedirs(OUT_MASK_DIR, exist_ok=True)

PATCH_SIZE        = 256
SEARCH_STRIDE     = 128   # step size when scanning for candidates
FG_CLASSES        = [2, 3, 4]
MIN_FG_RATIO      = 0.03  # patch must be ≥5% foreground (classes 2/3/4)
MAX_BG_RATIO      = 0.60  # patch must be ≤60% unannotated (class 0)
MAX_PATCHES_PER_TILE = 4  # take at most N patches per 2048 tile
MIN_NECROSIS_PIXELS  = 50     # a patch with any meaningful necrosis is valuable
MIN_PATCH_DISTANCE   = 128  

In [22]:
# classes of interest
FOREGROUND_CLASSES = [2, 3, 4]

def is_valid_patch(mask_patch):
    """
    Keep patch if it contains at least one meaningful class (2/3/4)
    """
    return np.any(np.isin(mask_patch, FOREGROUND_CLASSES))

In [23]:
def patch_score(mask):
    """
    Score a candidate patch. Higher = better.
    - Rewards foreground classes 2, 3, 4 (tumor tissue)
    - Penalises class 0 (unannotated / background)
    - Class 1 ("other") is annotated tissue — neutral, not penalised
    """
    total         = mask.size
    fg_ratio      = np.isin(mask, [2, 3, 4]).sum() / total
    bg_ratio      = (mask == 0).sum() / total
    necrosis_bonus = (mask == 4).sum() / total * 2.0  # heavily reward necrosis
    return fg_ratio - 0.3 * bg_ratio + necrosis_bonus

In [24]:
def extract_best_patch(img, mask):
    h, w = mask.shape

    best_score = -1
    best_img = None
    best_mask = None

    # stride smaller gives better search but slower
    stride = 128  

    for y in range(0, h - PATCH_SIZE + 1, stride):
        for x in range(0, w - PATCH_SIZE + 1, stride):

            m_patch = mask[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            score = patch_score(m_patch)

            if score > best_score:
                best_score = score
                best_mask = m_patch
                best_img = img[y:y+PATCH_SIZE, x:x+PATCH_SIZE]

    return best_img, best_mask


def is_too_close(y, x, accepted, min_dist):
    """Return True if (y, x) is within min_dist pixels of any accepted patch."""
    for (ay, ax) in accepted:
        if abs(y - ay) < min_dist and abs(x - ax) < min_dist:
            return True
    return False

In [25]:

def extract_patches_from_tile(img, mask):
    """
    Scan a 2048×2048 tile and return up to MAX_PATCHES_PER_TILE
    good (img_patch, mask_patch) pairs, ranked by score.
    """
    h, w = mask.shape
    candidates = []

    for y in range(0, h - PATCH_SIZE + 1, SEARCH_STRIDE):
        for x in range(0, w - PATCH_SIZE + 1, SEARCH_STRIDE):
            m = mask[y:y+PATCH_SIZE, x:x+PATCH_SIZE]

            total    = m.size
            fg_ratio = np.isin(m, FG_CLASSES).sum() / total
            bg_ratio = (m == 0).sum() / total

            # hard filters first (cheap)
            has_necrosis = (m == 4).sum() >= MIN_NECROSIS_PIXELS
            if not has_necrosis:
                if fg_ratio < MIN_FG_RATIO:   # normal threshold for non-necrosis patches
                    continue
            if bg_ratio > MAX_BG_RATIO:
                continue

            score = patch_score(m)
            candidates.append((score, y, x))

    # sort best-first
    candidates.sort(key=lambda t: t[0], reverse=True)

    selected       = []   # (img_patch, mask_patch)
    accepted_positions = []

    for score, y, x in candidates:
        if len(selected) >= MAX_PATCHES_PER_TILE:
            break

        # spatial diversity check — skip near-duplicates
        if is_too_close(y, x, accepted_positions, MIN_PATCH_DISTANCE):
            continue

        selected.append((
            img[y:y+PATCH_SIZE,  x:x+PATCH_SIZE],
            mask[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
        ))
        accepted_positions.append((y, x))

    return selected

In [ ]:
# --- main loop ---
files = sorted([
    f for f in os.listdir(MASK_PATCH_DIR)
    if f.endswith(".png") and not f.startswith(".")
])

total_saved = 0
stem_count  = {}   # track how many patches came from each tile

for fname in tqdm(files):
    img_path  = os.path.join(IMG_PATCH_DIR, fname)
    mask_path = os.path.join(MASK_PATCH_DIR, fname)

    img  = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if img is None or mask is None:
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    patches = extract_patches_from_tile(img, mask)

    stem = os.path.splitext(fname)[0]
    stem_count[stem] = len(patches)

    for i, (img_patch, mask_patch) in enumerate(patches):
        out_name = f"{stem}_p{i:02d}.png"
        cv2.imwrite(
            os.path.join(OUT_IMG_DIR,  out_name),
            cv2.cvtColor(img_patch, cv2.COLOR_RGB2BGR)
        )
        cv2.imwrite(
            os.path.join(OUT_MASK_DIR, out_name),
            mask_patch
        )
        total_saved += 1

print(f"\nDone. Saved {total_saved} patches from {len(files)} tiles.")
print(f"Avg patches per tile: {total_saved / max(len(files), 1):.1f}")
print(f"Tiles with 0 patches: {sum(1 for v in stem_count.values() if v == 0)}")

# quick class distribution check on a random sample
sample_masks = list(stem_count.keys())[:20]
print("\nClass pixel distribution (sample of up to 20 tiles' output):")
counts = np.zeros(5, dtype=np.int64)
for stem in sample_masks:
    for i in range(stem_count.get(stem, 0)):
        mp = cv2.imread(os.path.join(OUT_MASK_DIR, f"{stem}_p{i:02d}.png"),
                        cv2.IMREAD_GRAYSCALE)
        if mp is not None:
            for c in range(5):
                counts[c] += (mp == c).sum()

total_px = counts.sum()
class_names = ["unannotated", "other", "non-inv epi", "inv epi", "necrosis"]
for c, name in enumerate(class_names):
    print(f"  Class {c} ({name}): {counts[c]/total_px*100:.1f}%")

100%|██████████| 2941/2941 [07:39<00:00,  6.40it/s]


Done. Saved 7448 patches from 2941 tiles.
Avg patches per tile: 2.5
Tiles with 0 patches: 1050

Class pixel distribution (sample of up to 20 tiles' output):
  Class 0 (unannotated): 0.8%
  Class 1 (other): 42.6%
  Class 2 (non-inv epi): 38.0%
  Class 3 (inv epi): 18.7%
  Class 4 (necrosis): 0.0%


In [31]:
# quick class distribution check on a random sample
sample_masks = list(stem_count.keys())[:]
print("\nClass pixel distribution (sample of up to 20 tiles' output):")
counts = np.zeros(5, dtype=np.int64)
for stem in sample_masks:
    for i in range(stem_count.get(stem, 0)):
        mp = cv2.imread(os.path.join(OUT_MASK_DIR, f"{stem}_p{i:02d}.png"),
                        cv2.IMREAD_GRAYSCALE)
        if mp is not None:
            for c in range(5):
                counts[c] += (mp == c).sum()

total_px = counts.sum()
class_names = ["unannotated", "other", "non-inv epi", "inv epi", "necrosis"]
for c, name in enumerate(class_names):
    print(f"  Class {c} ({name}): {counts[c]/total_px*100:.1f}%")


Class pixel distribution (sample of up to 20 tiles' output):
  Class 0 (unannotated): 5.2%
  Class 1 (other): 7.7%
  Class 2 (non-inv epi): 42.1%
  Class 3 (inv epi): 27.9%
  Class 4 (necrosis): 16.9%


In [10]:
# get files
files = sorted([f for f in os.listdir(MASK_PATCH_DIR)
    if f.endswith(".png") and not f.startswith(".")])

for fname in tqdm(files):

    img_path = os.path.join(IMG_PATCH_DIR, fname)
    mask_path = os.path.join(MASK_PATCH_DIR, fname)

    img = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if img is None or mask is None:
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    best_img, best_mask = extract_best_patch(img, mask)

    # skip empty/invalid patches
    if best_mask is None:
        continue

    # enforce minimum tumor presence
    fg = ((best_mask == 2) | (best_mask == 3) | (best_mask == 4)).sum()
    fg_ratio = fg / (PATCH_SIZE * PATCH_SIZE)

    if fg_ratio < 0.05:   # at least 5% tumor tissue
        continue

    cv2.imwrite(os.path.join(OUT_IMG_DIR, fname), cv2.cvtColor(best_img, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(OUT_MASK_DIR, fname), best_mask)

100%|██████████| 2941/2941 [05:40<00:00,  8.65it/s]


In [ ]:
# get files
files = sorted([f for f in os.listdir(MASK_PATCH_DIR)
    if f.endswith(".png") and not f.startswith(".")])

kept = 0
total = 0

for f in tqdm(files):

    img_path = os.path.join(IMG_PATCH_DIR, f)
    mask_path = os.path.join(MASK_PATCH_DIR, f)

    img = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if img is None or mask is None:
        continue

    h, w = mask.shape

    # slide over image
    for y in range(0, h - PATCH_SIZE + 1, STRIDE):
        for x in range(0, w - PATCH_SIZE + 1, STRIDE):

            img_patch = img[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            mask_patch = mask[y:y+PATCH_SIZE, x:x+PATCH_SIZE]

            total += 1

            # --- filter condition ---
            if not is_valid_patch(mask_patch):
                continue

            # save
            base = f.replace(".png", f"_{y}_{x}.png")

            cv2.imwrite(os.path.join(OUT_IMG_DIR, base), img_patch)
            cv2.imwrite(os.path.join(OUT_MASK_DIR, base), mask_patch)

            kept += 1

print(f"\nTotal patches scanned: {total}")
print(f"Kept patches: {kept}")
print(f"Retention rate: {kept / total:.3f}")

Processing patient100_wsi1...
Saved 8 patches for patient100_wsi1.
Processing patient101_wsi1...
Saved 4 patches for patient101_wsi1.
Processing patient101_wsi2...
Saved 1 patches for patient101_wsi2.
Processing patient102_wsi1...
Saved 0 patches for patient102_wsi1.
Processing patient103_wsi1...
Reached max patches for patient103_wsi1. Stopping patching.
Processing patient104_wsi1...
Reached max patches for patient104_wsi1. Stopping patching.
Processing patient105_wsi1...
Saved 7 patches for patient105_wsi1.
Processing patient106_wsi1...
Saved 2 patches for patient106_wsi1.
Processing patient106_wsi2...
Saved 5 patches for patient106_wsi2.
Processing patient107_wsi1...
Saved 6 patches for patient107_wsi1.
Processing patient108_wsi1...
Reached max patches for patient108_wsi1. Stopping patching.
Processing patient108_wsi2...
Reached max patches for patient108_wsi2. Stopping patching.
Processing patient109_wsi1...
Reached max patches for patient109_wsi1. Stopping patching.
Processing pat

In [ ]:
# Add _0000 to end of file names

folder = r"/Volumes/LabData/BMED6460/beetle-master/data/images/development/output/beetle-patches/img_patches-256"

for filename in os.listdir(folder):
    if filename.startswith("."):
        continue
    name, ext = os.path.splitext(filename)
    new_name = f"{name}_0000{ext}"
    os.rename(
        os.path.join(folder, filename),
        os.path.join(folder, new_name)
    )

print("Done.")

Done.


In [ ]:
# Remove _0000 from end of file names

folder = r"/Volumes/LabData/BMED6460/beetle-master/data/images/development/output/beetle-patches/img_patches-256"

for filename in os.listdir(folder):
    if filename.startswith("."):
        continue
    name, ext = os.path.splitext(filename)
    if name.endswith("_0000"):
        new_name = f"{name[:-5]}{ext}"  # strip last 5 characters ("_0000")
        os.rename(
            os.path.join(folder, filename),
            os.path.join(folder, new_name)
        )

print("Done.")

Done.
